# Notebook 04 — Feature Engineering

This notebook engineers all features required for the Reinforcement Learning (RL) simulation environment.

**Input**: `data/processed/master_panel.csv` (9,059 rows × 51 columns)  
**Outputs**:
- `data/processed/master_panel_engineered.csv` — full dataset with all original + engineered features
- `data/processed/rl_training_data.csv` — streamlined dataset for the Gymnasium environment
- `data/processed/scaling_parameters.json` — robust scaling parameters per economy type
- `data/processed/calibration_profiles.csv` — 3×3 experimental matrix calibration table

**MDP State Space (7 variables)**: output growth, debt-to-GDP, primary balance, interest rate,
climate shock indicator, adaptation capital, risk premium.

**Experimental Design**: 3 economy types × 3 climate risk tiers = 9 agent profiles.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import os, subprocess, sys

# Ensure working directory is project root regardless of how kernel was launched
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print(f'Working directory: {os.getcwd()}')

def ensure_package(pkg):
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

ensure_package('statsmodels')

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.tsa.filters.hp_filter import hpfilter

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:.4f}'.format)

Path('data/processed').mkdir(parents=True, exist_ok=True)
print('Libraries loaded successfully.')

In [ ]:
# ── Load master panel ──────────────────────────────────────────────────────────
df = pd.read_csv('data/processed/master_panel.csv')
print(f'Loaded master panel: {df.shape[0]} rows × {df.shape[1]} columns')
assert df.shape[1] == 51, f'Expected 51 columns, got {df.shape[1]}'
print('\nColumn list:')
for i, c in enumerate(df.columns, 1):
    print(f'  {i:2d}. {c}')

df = df.sort_values(['iso3', 'year']).reset_index(drop=True)
print(f'\nSorted by iso3, year. Year range: {df["year"].min()}–{df["year"].max()}')
print(f'Countries: {df["iso3"].nunique()}')

---
## Step 1: Data Boundary Decisions

Before engineering any features, four scoping decisions that affect every subsequent calculation is noted 

- Years before 1990 have sparse coverage and pre-date the modern sovereign debt landscape.
- WEO projections (2025–2029) are model-generated, including them in historical statistics would contaminate calibration.
- The 4 unclassified countries block the 3×3 experimental matrix.
- Climate NaN ≠ missing. it means no disaster occurred, so zero-filling is conceptually correct.

In [ ]:
# ── Step 1a: Year range restriction & projection flag ──────────────────────────
print('=== STEP 1a: Year range restriction ===')

rows_before = len(df)
df['is_projection'] = df['year'] >= 2025
n_projection = df['is_projection'].sum()
print(f'Projection rows flagged (2025–2029): {n_projection}')

df = df[df['year'] >= 1990].reset_index(drop=True)
rows_dropped = rows_before - len(df)
print(f'Rows dropped (pre-1990): {rows_dropped}')
print(f'Working dataset shape after restriction: {df.shape}')
print(f'Year range retained: {df["year"].min()}–{df["year"].max()}')

In [ ]:
# ── Step 1b: Fix 4 unclassified countries ─────────────────────────────────────
print('=== STEP 1b: Assign missing economy_type ===')

manual_assignments = {
    'ETH': 'Developing',
    'VEN': 'Emerging Market',
    'WBG': 'Developing',
    'UVK': 'Emerging Market',
}

for iso, etype in manual_assignments.items():
    mask = df['iso3'] == iso
    n = mask.sum()
    df.loc[mask, 'economy_type'] = etype
    print(f'  {iso}: assigned "{etype}" ({n} rows)')

remaining_null = df['economy_type'].isna().sum()
print(f'\nRemaining economy_type NaNs: {remaining_null}')
print('\nEconomy type distribution:')
print(df['economy_type'].value_counts())

In [ ]:
# ── Step 1c: Fill climate NaNs with zero (within EM-DAT coverage: 2000–2024) ──
print('=== STEP 1c: Zero-fill climate columns (2000–2024) ===')

climate_cols = ['climate_disaster_count', 'climate_damage_000usd', 'climate_damage_adj_000usd']
emdat_mask = (df['year'] >= 2000) & (df['year'] <= 2024)

total_filled = 0
for col in climate_cols:
    n_before = df.loc[emdat_mask, col].isna().sum()
    df.loc[emdat_mask & df[col].isna(), col] = 0.0
    n_after = df.loc[emdat_mask, col].isna().sum()
    filled = n_before - n_after
    total_filled += filled
    print(f'  {col}: {filled} NaNs filled with 0 (still NaN outside window: {n_after})')

print(f'\nTotal values zero-filled: {total_filled}')

---
## Step 2: Debt Dynamics Features

Debt dynamics are central to both the **MDP state space** (`debt_to_gdp`) and the **reward
function** (quadratic debt penalty and interest burden).

### 2a — Year-on-year change in debt-to-GDP
Whether debt is rising or falling is as important as the level. The RL agent needs this trajectory
signal to understand whether its fiscal actions are working. Extreme single-year jumps (>50 pp)
are likely real events (debt restructurings, GDP collapses) and are flagged but retained.

### 2b — Interest payments proxy and debt service ratio
The identity `fiscal_balance = primary_balance − interest_payments` lets us recover interest
payments without a direct series. Normalised by revenue, this measures how much of every dollar
of government income is consumed by debt service — a direct input to the reward function's
interest burden penalty.

In [ ]:
# ── Step 2a: Year-on-year change in debt-to-GDP ───────────────────────────────
print('=== STEP 2a: Debt-to-GDP change ===')

df['debt_to_gdp_change'] = df.groupby('iso3')['debt_to_gdp'].diff()

print('Distribution summary:')
print(df['debt_to_gdp_change'].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]))

print('\n--- Extreme outliers (|change| > 50 pp in one year) ---')
outliers = df[df['debt_to_gdp_change'].abs() > 50][
    ['iso3', 'year', 'debt_to_gdp', 'debt_to_gdp_change']].sort_values('debt_to_gdp_change')
print(f'Count: {len(outliers)}')
print(outliers.to_string(index=False))
print('\nNote: flagged as likely real events, not data errors.')

print('\n--- Sanity check: USA ---')
print(df[df['iso3']=='USA'][['year','debt_to_gdp','debt_to_gdp_change']].query('year >= 2018').to_string(index=False))

In [ ]:
# ── Step 2b: Interest payments proxy and debt service ratio ───────────────────
print('=== STEP 2b: Interest payments proxy ===')

# fiscal_balance = primary_balance - interest_payments
# => interest_payments = primary_balance - fiscal_balance (positive = outflow)
df['interest_payments'] = -(df['weo_fiscal_balance'] - df['weo_primary_balance'])

df['debt_service_ratio'] = df['interest_payments'] / df['weo_govt_revenue']
# Cap implausible extremes from near-zero revenue
df.loc[df['debt_service_ratio'].abs() > 10, 'debt_service_ratio'] = np.nan

print('Interest payments distribution:')
print(df['interest_payments'].describe())
print(f'Non-null: {df["interest_payments"].notna().sum():,}')

print('\nDebt service ratio distribution:')
print(df['debt_service_ratio'].describe())
print(f'Non-null: {df["debt_service_ratio"].notna().sum():,}')

print('\n--- Sanity check: BRA (Emerging Market) ---')
print(df[df['iso3']=='BRA'][['year','weo_fiscal_balance','weo_primary_balance',
     'interest_payments','debt_service_ratio']].query('year >= 2015').to_string(index=False))

---
## Step 3: Fiscal Policy Features

These features map to the **primary balance state variable** and the **fiscal adjustment penalty**
in the reward function.

### 3a Fiscal impulse
The fiscal impulse measures the *active* change in fiscal stance, stripping out automatic
stabilisers (taxes falling and spending rising automatically in recessions). Using the
cyclically-adjusted (structural) balance where available gives a cleaner signal of deliberate
policy choices. A positive impulse = fiscal tightening; negative = fiscal loosening.

### 3b & 3c Tax revenue and government expenditure ratios
Fiscal capacity features that define a country's 'policy room'. High tax revenue means more
capacity to absorb climate shocks without catastrophic fiscal adjustment.

In [ ]:
# ── Step 3a: Fiscal impulse ───────────────────────────────────────────────────
print('=== STEP 3a: Fiscal impulse ===')

# Gold standard: change in cyclically-adjusted (structural) balance
df['fiscal_impulse_structural'] = df.groupby('iso3')['weo_structural_balance'].diff()

# Fallback: change in primary balance (includes automatic stabiliser effects)
df['fiscal_impulse_simple'] = df.groupby('iso3')['weo_primary_balance'].diff()

# Combined: structural where available, simple elsewhere
df['fiscal_impulse'] = df['fiscal_impulse_structural'].combine_first(df['fiscal_impulse_simple'])

n_structural = df['fiscal_impulse_structural'].notna().sum()
n_simple_only = (df['fiscal_impulse_structural'].isna() & df['fiscal_impulse_simple'].notna()).sum()
print(f'From structural balance (gold standard): {n_structural:,}')
print(f'From simple primary balance (fallback):   {n_simple_only:,}')
print(f'Total fiscal_impulse coverage: {df["fiscal_impulse"].notna().sum():,}')
print('\nFiscal impulse distribution:')
print(df['fiscal_impulse'].describe())
print('\nPositive = tightening (austerity); Negative = loosening (stimulus)')

In [ ]:
# ── Step 3b & 3c: Tax revenue and government expenditure coverage ──────────────
print('=== STEP 3b: Tax revenue-to-GDP coverage ===')
nn_3b = df['wdi_tax_revenue_gdp'].notna().sum()
pct_3b = nn_3b / len(df) * 100
print(f'wdi_tax_revenue_gdp non-null: {nn_3b:,} / {len(df):,} ({pct_3b:.1f}%)')

print('\n=== STEP 3c: Government expenditure-to-GDP coverage ===')
nn_3c = df['wdi_gov_expenditure_gdp'].notna().sum()
pct_3c = nn_3c / len(df) * 100
print(f'wdi_gov_expenditure_gdp non-null: {nn_3c:,} / {len(df):,} ({pct_3c:.1f}%)')

---
## Step 4: Climate Risk Features

These features map to the **climate shock indicator** state variable and the **climate damage penalty** in the reward function. The RL agent must learn to build fiscal buffers in anticipation of climate-driven expenditure shocks.

### 4a Climate damage as % of GDP
Raw damage in USD is not comparable across countries. Normalising by GDP makes it scale-free and directly measures the fiscal impact of disasters.

### 4b & 4c Rolling averages of damage and frequency
Disaster data is extremely volatile year-to-year. A 5-year rolling average gives the agent a smoother signal of *underlying* climate stress rather than responding to individual events.

### 4d Climate vulnerability × debt interaction
A country that is both highly climate-vulnerable AND highly indebted faces a compounding risk, disasters shrink the tax base and simultaneously force emergency spending pushing debt higher precisely when fiscal space is smallest. 

In [ ]:
# ── Step 4a: Climate damage as % of GDP ──────────────────────────────────────
print('=== STEP 4a: Climate damage as % of GDP ===')

# Unit check
print('Unit check — weo_gdp_current vs wdi_gdp_current_usd (USA 2018–2023):')
print(df[df['iso3']=='USA'][['year','weo_gdp_current','wdi_gdp_current_usd']]
      .query('year >= 2018 and year <= 2023').to_string(index=False))
print('=> weo_gdp_current is in billions USD')
print('=> wdi_gdp_current_usd is in USD')
print('=> Using wdi_gdp_current_usd as primary; fallback: weo_gdp_current * 1e9')

# GDP denominator in USD
gdp_usd = df['wdi_gdp_current_usd'].combine_first(df['weo_gdp_current'] * 1e9)

# climate_damage_adj_000usd is in thousands of USD
df['climate_damage_gdp_pct'] = (df['climate_damage_adj_000usd'] * 1000) / gdp_usd * 100

# Verify: zero damage rows with available GDP should yield exactly 0.0
zero_dam = (df['climate_damage_adj_000usd'] == 0) & gdp_usd.notna()
assert (df.loc[zero_dam, 'climate_damage_gdp_pct'] == 0.0).all()

print('\nClimate damage % GDP distribution:')
print(df['climate_damage_gdp_pct'].describe(percentiles=[0.5, 0.9, 0.99, 0.999]))

print('\nTop 20 country-years by climate damage (% GDP):')
top20 = (df[['iso3','year','climate_damage_gdp_pct','country_name']]
         .dropna(subset=['climate_damage_gdp_pct'])
         .query('climate_damage_gdp_pct > 0')
         .nlargest(20, 'climate_damage_gdp_pct'))
print(top20.to_string(index=False))

In [ ]:
# ── Step 4b: 5-year rolling mean of climate damage % GDP ─────────────────────
print('=== STEP 4b: 5-year rolling average — climate damage % GDP ===')

df['climate_damage_gdp_pct_5yr'] = (
    df.groupby('iso3')['climate_damage_gdp_pct']
    .transform(lambda x: x.rolling(5, min_periods=1).mean())
)
print(f'Non-null: {df["climate_damage_gdp_pct_5yr"].notna().sum():,}')
print(df['climate_damage_gdp_pct_5yr'].describe())

# ── Step 4c: 5-year rolling mean of disaster count ───────────────────────────
print('\n=== STEP 4c: 5-year rolling average — disaster frequency ===')

df['climate_disaster_freq_5yr'] = (
    df.groupby('iso3')['climate_disaster_count']
    .transform(lambda x: x.rolling(5, min_periods=1).mean())
)
print(f'Non-null: {df["climate_disaster_freq_5yr"].notna().sum():,}')
print(df['climate_disaster_freq_5yr'].describe())

print('\n--- Sanity check: KEN (Developing) ---')
print(df[df['iso3']=='KEN'][['year','climate_disaster_count','climate_damage_gdp_pct',
     'climate_damage_gdp_pct_5yr','climate_disaster_freq_5yr']].query('year >= 2010').to_string(index=False))

In [ ]:
# ── Step 4d: Climate vulnerability × debt interaction ────────────────────────
print('=== STEP 4d: Climate vulnerability × debt interaction ===')

df['climate_debt_interaction'] = df['ndgain_vulnerability'] * df['debt_to_gdp']

print('Distribution summary:')
print(df['climate_debt_interaction'].describe())
print(f'Non-null: {df["climate_debt_interaction"].notna().sum():,}')
print('\nTop 10 most stressed country-years (highest interaction):')
top_stress = (df[['iso3','year','ndgain_vulnerability','debt_to_gdp',
                  'climate_debt_interaction','economy_type']]
              .dropna(subset=['climate_debt_interaction'])
              .nlargest(10, 'climate_debt_interaction'))
print(top_stress.to_string(index=False))

---
## Step 5: Macroeconomic Context Features

### 5a Implicit and real interest rate
The **implicit interest rate** recovers the effective cost of government borrowing from the income statement. The **real interest rate** subtracts inflation, giving the rate that matters for debt sustainability real debt service cost relative to real economic capacity.

### 5b Interest rate–growth differential (r − g)
Blanchard (2019): when r < g, debt can remain sustainable even with primary deficits, when r > g, debt-to-GDP rises autonomously. This is the single most important variable in sovereign debt sustainability analysis.

### 5c Output gap
The output gap determines whether fiscal deficits reflect automatic stabilisers (cyclical) or structural policy choices. WEO data is used where available (mostly advanced economies), the HP filter (λ=6.25, Ravn-Uhlig annual) fills the rest.

In [ ]:
# ── Step 5a: Implicit and real interest rate ──────────────────────────────────
print('=== STEP 5a: Implicit interest rate on government debt ===')

# Both interest_payments and debt_to_gdp are % of GDP, so their ratio is a rate
df['implicit_interest_rate'] = (df['interest_payments'] / df['debt_to_gdp']) * 100

# Cap implausible values
df.loc[df['implicit_interest_rate'] < 0, 'implicit_interest_rate'] = np.nan
df.loc[df['implicit_interest_rate'] > 100, 'implicit_interest_rate'] = np.nan

print('Implicit interest rate distribution:')
print(df['implicit_interest_rate'].describe())
cov_implicit = df['implicit_interest_rate'].notna().mean() * 100
print(f'Coverage: {cov_implicit:.1f}%')

df['real_interest_rate'] = df['implicit_interest_rate'] - df['inflation']

print('\nReal interest rate distribution:')
print(df['real_interest_rate'].describe())
cov_real = df['real_interest_rate'].notna().mean() * 100
print(f'Coverage: {cov_real:.1f}%')

if cov_real < 50:
    print(f'\nWARNING: real_interest_rate coverage is {cov_real:.1f}% (<50%).')
    print('Fallback: implicit_interest_rate will be used for the RL state space.')
else:
    print(f'\nCoverage adequate ({cov_real:.1f}%). Using real_interest_rate.')

In [ ]:
# ── Step 5b: Interest rate – growth differential (r − g) ─────────────────────
print('=== STEP 5b: r − g differential ===')

df['r_minus_g'] = df['real_interest_rate'] - df['gdp_growth']

print('Overall distribution:')
print(df['r_minus_g'].describe())
print(f'Coverage: {df["r_minus_g"].notna().mean()*100:.1f}%')

print('\nMedian r−g by economy type (historical, non-projection):')
for etype in ['Advanced', 'Emerging Market', 'Developing']:
    sub = df[(df['economy_type'] == etype) & df['r_minus_g'].notna() & ~df['is_projection']]
    print(f'  {etype}: median={sub["r_minus_g"].median():.2f}, mean={sub["r_minus_g"].mean():.2f}, n={len(sub):,}')

print('\n--- Sanity check: USA, BRA, KEN (2015–2023) ---')
print(df[df['iso3'].isin(['USA','BRA','KEN'])][['iso3','year','real_interest_rate','gdp_growth','r_minus_g']]
      .query('year >= 2015 and year <= 2023').to_string(index=False))

In [ ]:
# ── Step 5c: Output gap (WEO primary; HP filter fallback) ─────────────────────
print('=== STEP 5c: Output gap ===')
print(f'WEO output gap: {df["weo_output_gap"].notna().sum():,} rows ({df["weo_output_gap"].notna().mean()*100:.1f}%)')

hp_gap_records = {}
n_hp_ok = 0
n_hp_short = 0

for iso, grp in df.groupby('iso3'):
    hist = grp[~grp['is_projection'] & grp['weo_gdp_constant'].notna()].sort_values('year')
    if len(hist) < 15:
        n_hp_short += 1
        continue
    try:
        log_gdp = np.log(hist['weo_gdp_constant'].values.astype(float))
        _, trend = hpfilter(log_gdp, lamb=6.25)
        gap = (log_gdp - trend) * 100
        for idx_row, g in zip(hist.index, gap):
            hp_gap_records[idx_row] = g
        n_hp_ok += 1
    except Exception as e:
        print(f'  HP filter failed for {iso}: {e}')

print(f'HP filter applied: {n_hp_ok} countries | skipped (<15 years): {n_hp_short}')

df['hp_output_gap'] = pd.Series(hp_gap_records)
df['output_gap'] = df['weo_output_gap'].combine_first(df['hp_output_gap'])

n_weo = df['weo_output_gap'].notna().sum()
n_hp = (df['weo_output_gap'].isna() & df['hp_output_gap'].notna()).sum()
print(f'output_gap — WEO: {n_weo:,} | HP filter: {n_hp:,} | total: {df["output_gap"].notna().sum():,}')
print(f'Final coverage: {df["output_gap"].notna().mean()*100:.1f}%')
print('\nOutput gap distribution:')
print(df['output_gap'].describe())

---
## Step 6: Lagged Features

Fiscal policy effects are not instantaneous a primary balance improvement today reduces debt-to-GDP next year. Lagged features give the RL agent **trajectory information** not just
where the country is now, but where it was. This helps the agent distinguish countries on improving vs deteriorating debt paths that happen to share the same current debt level.

The first 1–2 years per country will be NaN by construction.

In [ ]:
# ── Step 6: Lagged features ───────────────────────────────────────────────────
print('=== STEP 6: Lagged features ===')

lag_specs = [
    ('debt_to_gdp',            1, 'debt_to_gdp_lag1'),
    ('debt_to_gdp',            2, 'debt_to_gdp_lag2'),
    ('gdp_growth',             1, 'gdp_growth_lag1'),
    ('weo_primary_balance',    1, 'primary_balance_lag1'),
    ('climate_damage_gdp_pct', 1, 'climate_damage_gdp_pct_lag1'),
]

for source_col, periods, new_col in lag_specs:
    df[new_col] = df.groupby('iso3')[source_col].shift(periods)
    print(f'  {new_col}: {df[new_col].notna().sum():,} non-null values')

# Flag short time series
ts_length = df.groupby('iso3')['year'].count()
short_countries = ts_length[ts_length < 10].index.tolist()
print(f'\nCountries with <10 years of data (lags may be unreliable): {len(short_countries)}')
if short_countries:
    print(f'  {short_countries}')
    print('  NOTE: retained in dataset — user to decide exclusion.')

print('\n--- Sanity check: USA ---')
print(df[df['iso3']=='USA'][['year','debt_to_gdp','debt_to_gdp_lag1','debt_to_gdp_lag2',
     'gdp_growth','gdp_growth_lag1']].query('year >= 2018').to_string(index=False))

---
## Step 7: Climate Risk Tier Classification

Each country is assigned to one of three climate risk tiers to complete the 3×3 experimental
matrix. We use **ND-GAIN vulnerability** averaged over the most recent 5 years of available
data, split into terciles.

**Why terciles?** They guarantee exactly one-third of countries in each tier, ensuring
adequate observations in all 9 cells regardless of the vulnerability distribution's shape.
Cells with fewer than 5 countries are flagged for methodological discussion.

In [ ]:
# ── Step 7: Climate risk tier classification ──────────────────────────────────
print('=== STEP 7: Climate risk tier classification ===')

# Per-country mean vulnerability over most recent 5 years with data
vuln_recent = (
    df[df['ndgain_vulnerability'].notna()]
    .sort_values(['iso3', 'year'])
    .groupby('iso3')
    .apply(lambda g: g.tail(5)['ndgain_vulnerability'].mean(), include_groups=False)
    .rename('vuln_recent_mean')
    .reset_index()
)

print(f'Countries with vulnerability data: {len(vuln_recent)}')

t33 = vuln_recent['vuln_recent_mean'].quantile(0.333)
t67 = vuln_recent['vuln_recent_mean'].quantile(0.667)
print(f'Tercile thresholds — 33rd pct: {t33:.4f} | 67th pct: {t67:.4f}')

def assign_tier(v):
    if v <= t33: return 'Low'
    elif v <= t67: return 'Medium'
    else: return 'High'

vuln_recent['climate_risk_tier'] = vuln_recent['vuln_recent_mean'].apply(assign_tier)

df = df.merge(vuln_recent[['iso3', 'climate_risk_tier']], on='iso3', how='left')

print('\nCountries per climate risk tier:')
print(vuln_recent['climate_risk_tier'].value_counts())

# 3×3 experimental matrix
print('\n=== 3×3 Experimental Matrix (country counts) ===')
country_meta = df[['iso3','economy_type','climate_risk_tier']].drop_duplicates(subset='iso3')
xtab = pd.crosstab(country_meta['economy_type'], country_meta['climate_risk_tier'],
                   margins=True, margins_name='Total')
col_order = [c for c in ['Low','Medium','High','Total'] if c in xtab.columns]
row_order = [r for r in ['Advanced','Emerging Market','Developing','Total'] if r in xtab.index]
print(xtab.loc[row_order, col_order].to_string())

print('\nCells with < 5 countries:')
flagged = []
for etype in ['Advanced', 'Emerging Market', 'Developing']:
    for tier in ['Low', 'Medium', 'High']:
        n = country_meta[(country_meta['economy_type']==etype) & (country_meta['climate_risk_tier']==tier)].shape[0]
        if n < 5:
            flagged.append(f'  {etype} × {tier}: {n} countries')
print('\n'.join(flagged) if flagged else '  None — all cells have ≥ 5 countries.')

---
## Step 8: Normalisation for RL State Space

The Gymnasium environment requires all state variables to be on a comparable scale. We use
**robust scaling** (subtract median, divide by IQR) because:

1. Macroeconomic distributions are fat-tailed — outlier events (debt crises, hyperinflation)
   are real and important, but z-score standardisation would compress their signal.
2. Robust scaling is much less sensitive to extreme observations.
3. Scaling **within economy type** prevents spurious mappings (a 100% debt ratio means
   something different in an Advanced vs Developing economy).

The scaling parameters are saved to JSON — the Gymnasium environment must apply the *same*
scaling as training data.

In [ ]:
# ── Step 8: Robust normalisation for RL state space ───────────────────────────
print('=== STEP 8: Robust scaling of 7 state variables ===')

cov_real_check = df['real_interest_rate'].notna().mean() * 100
interest_rate_col = 'real_interest_rate' if cov_real_check >= 50 else 'implicit_interest_rate'
print(f'Interest rate source: {interest_rate_col} (real_interest_rate coverage: {cov_real_check:.1f}%)')

state_variable_map = {
    'state_output_growth':      'gdp_growth',
    'state_debt_to_gdp':        'debt_to_gdp',
    'state_primary_balance':    'weo_primary_balance',
    'state_interest_rate':      interest_rate_col,
    'state_climate_shock':      'climate_damage_gdp_pct_5yr',
    'state_adaptation_capital': 'ndgain_readiness',
    'state_risk_premium':       'r_minus_g',
}

hist_mask = ~df['is_projection']
economy_types = ['Advanced', 'Emerging Market', 'Developing']
scaling_params = {}

# Build lookup table: (state_col, economy_type) -> (median, iqr)
scale_lookup = {}  # {(state_col, etype): (median, iqr)}

for state_col, source_col in state_variable_map.items():
    scaling_params[state_col] = {}
    for etype in economy_types:
        mask = hist_mask & (df['economy_type'] == etype) & df[source_col].notna()
        vals = df.loc[mask, source_col]
        if len(vals) < 10:
            print(f'  WARNING: {state_col}/{etype} has {len(vals)} obs — using global stats')
            vals = df.loc[hist_mask & df[source_col].notna(), source_col]
        q25, q75 = vals.quantile(0.25), vals.quantile(0.75)
        iqr = q75 - q25
        if iqr == 0:
            iqr = vals.std()
        median = vals.median()
        scaling_params[state_col][etype] = {'median': float(median), 'iqr': float(iqr)}
        scale_lookup[(state_col, etype)] = (median, iqr)

# Vectorised application: map each row's economy_type to its scaling params
for state_col, source_col in state_variable_map.items():
    medians = df['economy_type'].map({et: scale_lookup[(state_col, et)][0] for et in economy_types})
    iqrs    = df['economy_type'].map({et: scale_lookup[(state_col, et)][1] for et in economy_types})
    df[state_col] = (df[source_col] - medians) / iqrs

print('\nScaled state variable summaries (median ≈ 0, IQR ≈ 1):')
for state_col in state_variable_map:
    s = df[state_col].describe()
    print(f'  {state_col}: mean={s["mean"]:.3f}, std={s["std"]:.3f}, '
          f'min={s["min"]:.2f}, max={s["max"]:.2f}, non-null={int(s["count"]):,}')

# Save scaling parameters
with open('data/processed/scaling_parameters.json', 'w') as f:
    json.dump({'interest_rate_source': interest_rate_col,
               'state_variable_map': state_variable_map,
               'parameters': scaling_params}, f, indent=2)
print('\nScaling parameters → data/processed/scaling_parameters.json')

---
## Step 9: Country Profile Calibration Table

The calibration table bridges historical data and the RL simulation environment. For each of
the 9 cells in the 3×3 matrix, we compute the median and IQR of all 7 state variables using
only historical observations (2000–2024, excluding projections).

The RL environment will be initialised using these profiles — the agent's starting state
distribution will reflect historical reality for each economy-climate combination.

In [ ]:
# ── Step 9: Country profile calibration table ─────────────────────────────────
print('=== STEP 9: Calibration profiles for 3×3 experimental matrix ===')

calib_df = df[(df['year'] >= 2000) & (df['year'] <= 2024) & ~df['is_projection']].copy()

state_src = {
    'output_growth':      'gdp_growth',
    'debt_to_gdp':        'debt_to_gdp',
    'primary_balance':    'weo_primary_balance',
    'interest_rate':      interest_rate_col,
    'climate_shock':      'climate_damage_gdp_pct_5yr',
    'adaptation_capital': 'ndgain_readiness',
    'risk_premium':       'r_minus_g',
}

rows = []
for etype in ['Advanced', 'Emerging Market', 'Developing']:
    for tier in ['Low', 'Medium', 'High']:
        mask = (calib_df['economy_type'] == etype) & (calib_df['climate_risk_tier'] == tier)
        sub = calib_df[mask]
        row = {'economy_type': etype, 'climate_risk_tier': tier,
               'n_countries': sub['iso3'].nunique(), 'n_observations': len(sub)}
        for sv, src in state_src.items():
            vals = sub[src].dropna()
            if len(vals) > 0:
                row[f'{sv}_median'] = vals.median()
                row[f'{sv}_p25']    = vals.quantile(0.25)
                row[f'{sv}_p75']    = vals.quantile(0.75)
            else:
                row[f'{sv}_median'] = row[f'{sv}_p25'] = row[f'{sv}_p75'] = np.nan
        rows.append(row)

calib_table = pd.DataFrame(rows)
pd.set_option('display.float_format', '{:.3f}'.format)

print('\nMedian state variables per profile:')
median_cols = (['economy_type','climate_risk_tier','n_countries','n_observations'] +
               [f'{s}_median' for s in state_src])
print(calib_table[median_cols].to_string(index=False))

# Identify the stress scenario
if calib_table['debt_to_gdp_median'].notna().any():
    si = calib_table['debt_to_gdp_median'].idxmax()
    sr = calib_table.loc[si]
    print(f'\nStress scenario profile (highest median debt):')
    print(f'  {sr["economy_type"]} × {sr["climate_risk_tier"]} | debt-to-GDP: {sr["debt_to_gdp_median"]:.1f}%')

calib_table.to_csv('data/processed/calibration_profiles.csv', index=False)
print('\nCalibration profiles → data/processed/calibration_profiles.csv')
pd.set_option('display.float_format', '{:.4f}'.format)

---
## Step 10: Final Dataset Save and Summary

Two output files:
1. **`master_panel_engineered.csv`** — all 51 original columns plus all engineered features
2. **`rl_training_data.csv`** — streamlined input for the Gymnasium RL environment (7 normalised
   state variables + their un-normalised sources + identifiers)

In [ ]:
# ── Step 10: Save full engineered dataset ─────────────────────────────────────
print('=== STEP 10: Saving output files ===')

df.to_csv('data/processed/master_panel_engineered.csv', index=False)
print(f'Full engineered dataset: {df.shape} → data/processed/master_panel_engineered.csv')

# RL training dataset
rl_cols = [
    'iso3', 'year', 'economy_type', 'climate_risk_tier',
    # Normalised state variables
    'state_output_growth', 'state_debt_to_gdp', 'state_primary_balance',
    'state_interest_rate', 'state_climate_shock', 'state_adaptation_capital', 'state_risk_premium',
    # Un-normalised sources
    'gdp_growth', 'debt_to_gdp', 'weo_primary_balance',
    'climate_damage_gdp_pct_5yr', 'ndgain_readiness', 'r_minus_g',
    'is_projection',
]

# Add the interest rate un-normalised column (avoid duplicate if already in list)
if interest_rate_col not in rl_cols:
    rl_cols.insert(rl_cols.index('climate_damage_gdp_pct_5yr'), interest_rate_col)

rl_df = df[rl_cols].copy()
rl_df.to_csv('data/processed/rl_training_data.csv', index=False)
print(f'RL training dataset:     {rl_df.shape} → data/processed/rl_training_data.csv')

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print('=' * 62)
print('=== FEATURE ENGINEERING COMPLETE ===')
print('=' * 62)
print(f'Full engineered dataset:  {df.shape}  →  master_panel_engineered.csv')
print(f'RL training dataset:      {rl_df.shape}  →  rl_training_data.csv')
print(f'Scaling parameters:       saved to scaling_parameters.json')
print(f'Calibration profiles:     saved to calibration_profiles.csv')

print('\nExperimental matrix coverage (country counts):')
country_meta2 = df[['iso3','economy_type','climate_risk_tier']].drop_duplicates(subset='iso3')
xtab2 = pd.crosstab(country_meta2['economy_type'], country_meta2['climate_risk_tier'],
                    margins=True, margins_name='Total')
col2 = [c for c in ['Low','Medium','High','Total'] if c in xtab2.columns]
row2 = [r for r in ['Advanced','Emerging Market','Developing','Total'] if r in xtab2.index]
print(xtab2.loc[row2, col2].to_string())

print('\nState variable coverage in RL training data:')
state_vars = ['state_output_growth','state_debt_to_gdp','state_primary_balance',
              'state_interest_rate','state_climate_shock','state_adaptation_capital','state_risk_premium']
for sv in state_vars:
    n = rl_df[sv].notna().sum()
    pct = n / len(rl_df) * 100
    print(f'  {sv}: {n:,} ({pct:.1f}%)')

print('\nAll outputs confirmed. Notebook 04 complete.')